# **ST 554 Final Project**
---
Authored by: Jamie Loring

Collaborator: Dr. Justin Post (code from lecture videos & slides)

## **Setup**

The code below imports the required modules that are needed for this project.

In [1]:
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.ml.feature import SQLTransformer, Binarizer, VectorAssembler, OneHotEncoder, PCA, StandardScaler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql.functions import col
from pyspark.ml import Pipeline

The code below sets up our `spark` object and only allows errors to print out in the future (i.e., it suppresses warnings).

In [2]:
spark = SparkSession.builder.master('local[*]').appName('my_app').getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/28 17:26:49 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/28 17:26:49 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


The code below reads in the `power_ml_data.csv` file from the provided URL using `pandas`.

In [3]:
power_data = pd.read_csv("https://www4.stat.ncsu.edu/~online/datasets/power_ml_data.csv")
power_data.head()

,Temperature,Humidity,Wind_Speed,General_Diffuse_Flows,Diffuse_Flows,Power_Zone_1,Power_Zone_2,Power_Zone_3,Month,Hour
0,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.87538,20240.96386,1,0
1,6.414,74.5,0.083,0.070,0.085,29814.68354,19375.07599,20131.08434,1,0
2,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.68693,19668.43373,1,0
3,6.121,75.0,0.083,0.091,0.096,28228.86076,18361.09422,18899.27711,1,0
4,5.921,75.7,0.081,0.048,0.085,27335.69620,17872.34043,18442.40964,1,0


Now, we convert our `pandas dataframe` to a `spark` SQL style dataframe. This new object will be called `power_ML`. For better readability, throughout this project, I will display results using `.toPandas()` and `.head()` rather than `.show()`. It is important to note that this choice does not change `power_ML` back to a `pandas` or `pandas-on-spark` dataframe; rather, it just changes the way it is *displayed*.

In [5]:
power_ML = spark.createDataFrame(power_data)
power_ML.toPandas().head()

,Temperature,Humidity,Wind_Speed,General_Diffuse_Flows,Diffuse_Flows,Power_Zone_1,Power_Zone_2,Power_Zone_3,Month,Hour
0,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.87538,20240.96386,1,0
1,6.414,74.5,0.083,0.070,0.085,29814.68354,19375.07599,20131.08434,1,0
2,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.68693,19668.43373,1,0
3,6.121,75.0,0.083,0.091,0.096,28228.86076,18361.09422,18899.27711,1,0
4,5.921,75.7,0.081,0.048,0.085,27335.69620,17872.34043,18442.40964,1,0


We are going to treat the `Power_Zone_3` variable as our response variable. We want to imagine that we know that the `Power_Zone_3` reading is going to go offline in the future, and we need to be able to predict that value appropriately using the other variables in the dataset.

## **Fitting the Model**

We are going to fit an elastic net model using CV without a training & testing split. Before we can fit our model, we have to perform the required transformations using `MLlib` functions. All (nested) transformations will be saved as objects so 1) We can display our progress through the transformations and 2) We can check to make sure we do not miss any transformations in the final pipeline.

First, we need to check to see how the `Hour` column is stored. 

In [6]:
power_ML.dtypes

[('Temperature', 'double'),
 ('Humidity', 'double'),
 ('Wind_Speed', 'double'),
 ('General_Diffuse_Flows', 'double'),
 ('Diffuse_Flows', 'double'),
 ('Power_Zone_1', 'double'),
 ('Power_Zone_2', 'double'),
 ('Power_Zone_3', 'double'),
 ('Month', 'bigint'),
 ('Hour', 'bigint')]

Now that we have confirmed that it's not a `DoubleType`, we will need to use an SQL transformer to cast the `Hour` variable as a `DoubleType`.

In [7]:
cast_hour = SQLTransformer(
    statement = """
                SELECT *, CAST(Hour as DOUBLE) as Hour_Double_Type FROM __THIS__
                """
            )

tf_1 = cast_hour.transform(power_ML)
tf_1.toPandas().head()

,Temperature,Humidity,Wind_Speed,General_Diffuse_Flows,Diffuse_Flows,Power_Zone_1,Power_Zone_2,Power_Zone_3,Month,Hour,Hour_Double_Type
0,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.87538,20240.96386,1,0,0.0
1,6.414,74.5,0.083,0.070,0.085,29814.68354,19375.07599,20131.08434,1,0,0.0
2,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.68693,19668.43373,1,0,0.0
3,6.121,75.0,0.083,0.091,0.096,28228.86076,18361.09422,18899.27711,1,0,0.0
4,5.921,75.7,0.081,0.048,0.085,27335.69620,17872.34043,18442.40964,1,0,0.0


Next, we need to binarize the updated `Hour_Double_Type` column based on it being less than 6.5 or not. We are essentially creating a night vs. day variable.

In [8]:
binarize_hour = Binarizer(threshold = 6.5, inputCol = "Hour_Double_Type", outputCol = "Night_vs_Day")

tf_2 = binarize_hour.transform(tf_1)
tf_2.toPandas().head()

,Temperature,Humidity,Wind_Speed,General_Diffuse_Flows,Diffuse_Flows,Power_Zone_1,Power_Zone_2,Power_Zone_3,Month,Hour,Hour_Double_Type,Night_vs_Day
0,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.87538,20240.96386,1,0,0.0,0.0
1,6.414,74.5,0.083,0.070,0.085,29814.68354,19375.07599,20131.08434,1,0,0.0,0.0
2,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.68693,19668.43373,1,0,0.0,0.0
3,6.121,75.0,0.083,0.091,0.096,28228.86076,18361.09422,18899.27711,1,0,0.0,0.0
4,5.921,75.7,0.081,0.048,0.085,27335.69620,17872.34043,18442.40964,1,0,0.0,0.0


Continuing on, we need to one-hot encode the `Month` variable. Since `Month` is already an integer, we do _**not**_ need to use `StringIndexer()`.

In [10]:
encoder_OHE = OneHotEncoder(inputCols = ["Month"], outputCols = ["Month_OHE"])
model_OHE = encoder_OHE.fit(tf_2)

tf_3 = model_OHE.transform(tf_2)
tf_3.toPandas().head()

,Temperature,Humidity,Wind_Speed,General_Diffuse_Flows,Diffuse_Flows,Power_Zone_1,Power_Zone_2,Power_Zone_3,Month,Hour,Hour_Double_Type,Night_vs_Day,Month_OHE
0,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.87538,20240.96386,1,0,0.0,0.0,"(0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1,6.414,74.5,0.083,0.070,0.085,29814.68354,19375.07599,20131.08434,1,0,0.0,0.0,"(0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
2,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.68693,19668.43373,1,0,0.0,0.0,"(0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
3,6.121,75.0,0.083,0.091,0.096,28228.86076,18361.09422,18899.27711,1,0,0.0,0.0,"(0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
4,5.921,75.7,0.081,0.048,0.085,27335.69620,17872.34043,18442.40964,1,0,0.0,0.0,"(0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."


Our next step is to run a PCA fit on the following variables:
- `Temperature`
- `Humidity`
- `Wind_Speed`
- `General_Diffuse_Flows`
- `Diffuse_Flows`

We will also need to standarize our predictors using `StandardScaler()`.

**Note:** This will be a multi-step process, outlined with in-line comments in the code chunk below.

In [12]:
# use VectorAssembler() to place the above variables in a column together for use with the PCA() estimator
assembler_PCA = VectorAssembler(
                inputCols = ["Temperature", "Humidity", "Wind_Speed",
                             "General_Diffuse_Flows", "Diffuse_Flows"],
                outputCol = "features_for_PCA",
                handleInvalid = "keep"
             )

# update transformations
tf_4 = assembler_PCA.transform(tf_3)

# use StandardScaler() to standardize predictors
scaler_PCA = StandardScaler(inputCol = "features_for_PCA", outputCol = "features_for_PCA_scaled",
                            withStd = True, withMean = True)
# fit scaler_PCA
scaler_fit = scaler_PCA.fit(tf_4)

# update transformations
tf_5 = scaler_fit.transform(tf_4)

# run PCA, using 2 PC's in the transformation
PCA_func = PCA(k = 2, inputCol = "features_for_PCA_scaled", outputCol = "PCA_features_scaled")

# fit the PCA model
PCA_model = PCA_func.fit(tf_5)

# update transformations
tf_6 = PCA_model.transform(tf_5)

# show updated transformations
tf_6.toPandas().head()

,Temperature,Humidity,Wind_Speed,General_Diffuse_Flows,Diffuse_Flows,Power_Zone_1,Power_Zone_2,Power_Zone_3,Month,Hour,Hour_Double_Type,Night_vs_Day,Month_OHE,features_for_PCA,features_for_PCA_scaled,PCA_features_scaled
0,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.87538,20240.96386,1,0,0.0,0.0,"(0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[6.559, 73.8, 0.083, 0.051, 0.119]","[-2.1079477438809433, 0.3542085264292604, -0.7...","[2.0690743213463727, 0.8078678829472257]"
1,6.414,74.5,0.083,0.070,0.085,29814.68354,19375.07599,20131.08434,1,0,0.0,0.0,"(0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[6.414, 74.5, 0.083, 0.07, 0.085]","[-2.1328903699941857, 0.3991947174962608, -0.7...","[2.1029210638806544, 0.8152476222806389]"
2,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.68693,19668.43373,1,0,0.0,0.0,"(0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[6.313, 74.5, 0.08, 0.062, 0.1]","[-2.1502641992178924, 0.3991947174962608, -0.8...","[2.112066263379102, 0.8227151924829954]"
3,6.121,75.0,0.083,0.091,0.096,28228.86076,18361.09422,18899.27711,1,0,0.0,0.0,"(0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[6.121, 75.0, 0.083, 0.091, 0.096]","[-2.183291676554047, 0.4313277111155467, -0.79...","[2.14350318474222, 0.8329135817940959]"
4,5.921,75.7,0.081,0.048,0.085,27335.69620,17872.34043,18442.40964,1,0,0.0,0.0,"(0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[5.921, 75.7, 0.081, 0.048, 0.085]","[-2.217695298779209, 0.47631390218254716, -0.8...","[2.182444003662701, 0.8444681624812325]"


We now need to put our model predictors into a single column called features. We can do this with `VectorAssembler()`. We will also use `StandardScaler()` to standardize our features. Since `PCA_features_scaled` has already been standardized, we do not need to re-standardize it, so this will be a multi-step process.

In [14]:
# assemble features vector without PCA for scaling
assembler_features = VectorAssembler(
                          inputCols = ["Night_vs_Day", "Power_Zone_1",
                                       "Power_Zone_2", "Month_OHE"],
                          outputCol = "features_pre_scale",
                          handleInvalid = "keep"
                    )

# update transformations
tf_7 = assembler_features.transform(tf_6)

# use StandardScaler() to standardize features -- remember PCA_features_scaled is already standardized!
scaler_features = StandardScaler(inputCol = "features_pre_scale", outputCol = "features_scaled",
                                 withStd = True, withMean = True)

# fit scaler_features
scaler_fit_feat = scaler_features.fit(tf_7)

# update transformations
tf_8 = scaler_fit_feat.transform(tf_7)

# use VectorAssembler() again to combine PCA_features_scaled and features_scaled into one features column
assembler_features_final = VectorAssembler(
                                inputCols = ["PCA_features_scaled", "features_scaled"],
                                outputCol = "features",
                                handleInvalid = "keep"
                        )

# update transformations
tf_9 = assembler_features_final.transform(tf_8)

# show updated transformations
tf_9.toPandas().head()

,Temperature,Humidity,Wind_Speed,General_Diffuse_Flows,Diffuse_Flows,Power_Zone_1,Power_Zone_2,Power_Zone_3,Month,Hour,Hour_Double_Type,Night_vs_Day,Month_OHE,features_for_PCA,features_for_PCA_scaled,PCA_features_scaled,features_pre_scale,features_scaled,features
0,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.87538,20240.96386,1,0,0.0,0.0,"(0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[6.559, 73.8, 0.083, 0.051, 0.119]","[-2.1079477438809433, 0.3542085264292604, -0.7...","[2.0690743213463727, 0.8078678829472257]","(0.0, 34055.6962, 16128.87538, 0.0, 1.0, 0.0, ...","[-1.5544690531019132, 0.24130775579578978, -0....","[2.0690743213463727, 0.8078678829472257, -1.55..."
1,6.414,74.5,0.083,0.070,0.085,29814.68354,19375.07599,20131.08434,1,0,0.0,0.0,"(0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[6.414, 74.5, 0.083, 0.07, 0.085]","[-2.1328903699941857, 0.3991947174962608, -0.7...","[2.1029210638806544, 0.8152476222806389]","(0.0, 29814.68354, 19375.07599, 0.0, 1.0, 0.0,...","[-1.5544690531019132, -0.35350356900601565, -0...","[2.1029210638806544, 0.8152476222806389, -1.55..."
2,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.68693,19668.43373,1,0,0.0,0.0,"(0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[6.313, 74.5, 0.08, 0.062, 0.1]","[-2.1502641992178924, 0.3991947174962608, -0.8...","[2.112066263379102, 0.8227151924829954]","(0.0, 29128.10127, 19006.68693, 0.0, 1.0, 0.0,...","[-1.5544690531019132, -0.449798237837335, -0.3...","[2.112066263379102, 0.8227151924829954, -1.554..."
3,6.121,75.0,0.083,0.091,0.096,28228.86076,18361.09422,18899.27711,1,0,0.0,0.0,"(0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[6.121, 75.0, 0.083, 0.091, 0.096]","[-2.183291676554047, 0.4313277111155467, -0.79...","[2.14350318474222, 0.8329135817940959]","(0.0, 28228.86076, 18361.09422, 0.0, 1.0, 0.0,...","[-1.5544690531019132, -0.5759186911227896, -0....","[2.14350318474222, 0.8329135817940959, -1.5544..."
4,5.921,75.7,0.081,0.048,0.085,27335.69620,17872.34043,18442.40964,1,0,0.0,0.0,"(0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[5.921, 75.7, 0.081, 0.048, 0.085]","[-2.217695298779209, 0.47631390218254716, -0.8...","[2.182444003662701, 0.8444681624812325]","(0.0, 27335.6962, 17872.34043, 0.0, 1.0, 0.0, ...","[-1.5544690531019132, -0.7011869790980542, -0....","[2.182444003662701, 0.8444681624812325, -1.554..."


Finally, we rename the response variable, `Power_Zone_3`, as `label`. We will also use the SQL transformer to select the `features` vector that was created in the previous step.

In [15]:
label_and_features = SQLTransformer(
    statement = """
                SELECT features, Power_Zone_3 as label FROM __THIS__
                """
            )

# update transformations
tf_10 = label_and_features.transform(tf_9)
tf_10.toPandas().head()

,features,label
0,"[2.0690743213463727, 0.8078678829472257, -1.55...",20240.96386
1,"[2.1029210638806544, 0.8152476222806389, -1.55...",20131.08434
2,"[2.112066263379102, 0.8227151924829954, -1.554...",19668.43373
3,"[2.14350318474222, 0.8329135817940959, -1.5544...",18899.27711
4,"[2.182444003662701, 0.8444681624812325, -1.554...",18442.40964


We can now put all of our transformations into the pipeline. We will have 10 of them, plus our linear regression instance. They are:
- `cast_hour`
- `binarize_hour`
- `encoder_OHE`
- `assembler_PCA`
- `scaler_PCA`
- `PCA_func`
- `assembler_features`
- `scaler_features`
- `assembler_features_final`
- `label_and_features`
- and `lr`, which will be the instance of our `LinearRegression()`

**Note:** Only the transformers and estimators need to be put into the pipeline since the pipeline will automatically handle which ones need to be fit. The fit objects up to this point were only created to show the progress of the transformations and check their functionality, but they themselves do not go into the pipeline!

*Example:* `encoder_OHE` *goes into the pipeline rather than* `model_OHE`

In [16]:
lr = LinearRegression()
pipeline = Pipeline(stages = [cast_hour, binarize_hour, encoder_OHE, assembler_PCA,
                              scaler_PCA, PCA_func, assembler_features, scaler_features,
                              assembler_features_final, label_and_features, lr])

We are now ready to fit our elastic net model. We are provided with the following grid for `regParam` and `elasticNetParam`:
- `regParam`: 0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1
- `elasticNetParam`: 0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1

Since our `pipeline` object is already set up, we have the following remaining steps to complete in creating our model:
- build the parameter grid
- use cross-validation (5 folds) to choose the best combination of parameters using RMSE
- fit the model

**Note:** The use of `parallelism = 128` in `CrossValidator()` speeds up the runtime from about 12 minutes to about 6 minutes! Please be patient! Additionally, the red line that appears under the code chunk is *not* representative of an actual error, but rather staging outputs from the log.

In [17]:
# build parameter grid
paramGrid = ParamGridBuilder() \
            .addGrid(lr.regParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
            .addGrid(lr.elasticNetParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
            .build()


# set up cross validation with pipeline
crossval = CrossValidator(estimator = pipeline,
                          estimatorParamMaps = paramGrid,
                          evaluator = RegressionEvaluator(metricName = 'rmse'),
                          numFolds = 5,
                          seed = 44,
                          parallelism = 128)

# fit the model using cross-validation to choose the best model
cvModel = crossval.fit(power_ML)

Although the `.bestModel` attribute will store the best model as a result of cross-validation by default, we want to look to see what model is the best. The code below prints the RMSE for the model fit with its corresponding parameters, `regParam` and `elasticNetParam`. I then sort this output in ascending order so we can see the lowest RMSE first (along with its corresponding parameters).

In [18]:
my_list = []

for i in range(len(paramGrid)):
    my_list.append([cvModel.avgMetrics[i], paramGrid[i].values()])
    
my_list.sort(key=lambda x: x[0]) #specifies sorting by avgMetrics rather than the dict_values
my_list

[[2175.025614452874, dict_values([0.25, 0.5])],
 [2175.0335187645323, dict_values([0.5, 0.25])],
 [2175.0335454632104, dict_values([0.5, 0.05])],
 [2175.0341016574057, dict_values([0.5, 0.1])],
 [2175.0349079585444, dict_values([0.05, 0.5])],
 [2175.035462853383, dict_values([0.25, 0.1])],
 [2175.035512165499, dict_values([0.1, 0.95])],
 [2175.035587292835, dict_values([0.05, 0.98])],
 [2175.0356041252753, dict_values([0.05, 0.9])],
 [2175.0356236594257, dict_values([0.05, 0.95])],
 [2175.035731007239, dict_values([0.05, 0.99])],
 [2175.0357346658175, dict_values([0.05, 1.0])],
 [2175.035803359566, dict_values([0.05, 0.25])],
 [2175.035833255783, dict_values([0.1, 0.0])],
 [2175.035841619817, dict_values([0.1, 0.1])],
 [2175.0358417832153, dict_values([0.25, 0.0])],
 [2175.0358425776453, dict_values([0.05, 0.75])],
 [2175.03585093693, dict_values([0.05, 0.0])],
 [2175.035880512319, dict_values([0.0, 0.9])],
 [2175.0358805123224, dict_values([0.0, 0.25])],
 [2175.035880590477, dict_valu

Thus, the best multiple linear regression model based on cross-validation with RMSE is one with a regularization parameter of 0.25 and an elastic net parameter of 0.5. This tells us that we have a combination of L1 and L2 penalties; hence, we are fitting an elastic net model! We can now print the intercept and coefficients of this best model, as well as the CV error (with its corresponding tuning parameters).

In [19]:
# need to use indexing since the model is the last stage of the pipeline
print('Intercept:', cvModel.bestModel.stages[-1].intercept)
print('Coefficients (in Features Order):', cvModel.bestModel.stages[-1].coefficients)

print(' ')

print("Best regParam:", cvModel.bestModel.stages[-1]._java_obj.getRegParam())
print("Best elasticNetParam:", cvModel.bestModel.stages[-1]._java_obj.getElasticNetParam())
print('Resulting Best CV RMSE:', round(min(cvModel.avgMetrics), 4))

Intercept: 17831.197607816746
Coefficients (in Features Order): [281.3375940841669,-508.54885234099714,-933.3422279345779,4380.97367310121,582.7712870594947,0.0,1670.6436195006277,1521.0143011644382,1488.0485558207392,1932.6899471239308,1411.9841811335189,1784.2129277382987,3556.71978908298,2402.836302038804,373.595060553956,-54.722433929874555,504.521785878895]
 
Best regParam: 0.25
Best elasticNetParam: 0.5
Resulting Best CV RMSE: 2175.0256


We now report the training set RMSE, i.e., the RMSE on the full dataset, by using the fitted model as a transformer and evaluating on the entire training (full) dataset.

In [20]:
training_rmse = RegressionEvaluator(metricName = 'rmse').evaluate(cvModel.transform(power_ML))

print('Entire Training (Full) Dataset RMSE:', round(training_rmse, 5))

Entire Training (Full) Dataset RMSE: 2174.44967


The last step is to take the outputted transformations from the model (i.e., the predictions) and create a `residual` column defined as `label` - `prediction`. We will print out a resulting dataframe with the following columns:
- `residual`
- `label`
- `prediction`

In [21]:
# store previous model transformer in an object
tf_final = cvModel.transform(power_ML)

# create residual column
resids_label_preds = tf_final.withColumn("residual", col("label") - col("prediction"))
resids_label_preds.select("residual", "label", "prediction").toPandas()

,residual,label,prediction
0,-695.215712,20240.96386,20936.179572
1,1431.166976,20131.08434,18699.917364
2,1432.893079,19668.43373,18235.540651
3,1284.964277,18899.27711,17614.312833
4,1426.591996,18442.40964,17015.817644
...,...,...,...
47169,2469.850984,14780.31212,12310.461136
47170,2647.362365,14428.81152,11781.449155
47171,2633.732742,13806.48259,11172.749848
47172,2794.162807,13512.60504,10718.442233


## **Streaming**
In this part of the project, we will be using the model created in the previous section with streaming data!

### **Reading a Stream**
In my `jupyterhub` main directory, I have created a folder called `csv_files_final`. This folder will be used to read in the stream in the form of `.csv` files.

**Note:** Make sure this folder is empty before starting the stream!

The code below sets up the schema for the stream by using the schema from the original data as we did in the Homework 10 assignment.

In [22]:
myschema = power_ML.schema
power_stream = spark.readStream.option("header", "true").schema(myschema).csv("csv_files_final")

### **Transformations and Aggregations**

First, with the stream, we use the model transformer to obtain predictions from the incoming data. This can be accomplished by creating an object very similar to `tf_final`. The only difference is that rather than supplying `power_ML` -- our data object -- to the `transform()` method, we will supply the stream object -- `power_stream`. The newly created object will be called `power_transform`, and we can use this to create a `residual` column.

In [23]:
# use model transformer with stream
power_transform = cvModel.transform(power_stream)

# create residual column
pwr_tf_resids = power_transform.withColumn("residual", col("label") - col("prediction")) \
                               .select("residual", "label", "prediction")

Next, we can use the stream again, with another transformation on the original stream, by modifying the response variable (`Power_Zone_3`) to be called `label`. This will be done using the `.withColumnRenamed()` method, rather than an SQL Transformer, because we want to keep all the original variables as well.

In [24]:
label_rename = power_stream.withColumnRenamed("Power_Zone_3", "label")

We can now join the above previous transform with this stream based on the `label` variable. This will be done with an inner join via the `.join()` method.

In [25]:
stream_join = pwr_tf_resids.join(label_rename, "label", "inner")

### **Writing Step**
We are now ready to write the `stream_join`. We will write it to the console using the `append` output mode. The code below will start the query.

In [26]:
query = stream_join.writeStream.outputMode("append").format("console").start()

-------------------------------------------
Batch: 0
-------------------------------------------
+-----------+-------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|           residual|        prediction|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+-------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|17105.10121|  4725.615382895274|12379.485827104727|      25.61|   41.78|     0.085|                577.4|        613.7| 29146.22951| 17305.26316|    5|  16|
|16099.05528| 437.06508789037616|15661.990192109624|      12.28|    76.0|     4.916|                 0.07|        0.089| 25023.05085| 15169.60486|    2|   1|
|15777.19088|-3394.1064862630537|19171.297366263054|      15.25|    78.9|     0.091|                0.073|       

-------------------------------------------
Batch: 1
-------------------------------------------
+-----------+-------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|           residual|        prediction|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+-------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|23404.33735|  530.6106199253918| 22873.72673007461|      12.86|   55.36|     0.084|                23.41|        24.69|  39396.4557| 24503.34347|    1|  17|
|19323.87097| 1562.5017390891117| 17761.36923091089|      12.17|    84.2|     0.076|                0.088|         0.13| 32427.57447| 19778.04878|    3|  23|
|14260.64516| 373.11054500493265|13887.534614995067|        9.8|    87.9|     0.077|                0.044|       

-------------------------------------------
Batch: 2
-------------------------------------------
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|          residual|        prediction|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|13060.66869|-902.2109409024288| 13962.87963090243|      24.68|   63.72|     4.922|                 96.8|         81.7| 34686.03939|  21656.0166|   10|  16|
|19396.62651|1194.2705760643257|18202.355933935673|      17.12|    70.5|     4.916|                0.048|        0.133| 31376.20253| 19203.64742|    1|  23|
|24105.32915|-448.1907964059028|24553.519946405904|      27.88|   44.28|     4.906|                491.6|        48.52

-------------------------------------------
Batch: 3
-------------------------------------------
+-----------+-------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|           residual|        prediction|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+-------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|17087.39812| -4768.355699873617| 21855.75381987362|      29.51|   60.69|     4.921|                478.6|        185.2| 34342.64151| 21869.90496|    8|  16|
|24892.25806|  421.9036264135502| 24470.35443358645|      13.67|   51.13|     0.086|                0.242|        0.204| 42758.80851| 25785.36585|    3|  19|
| 16548.3871|  74.42780277183192|16473.959297228168|       10.6|    84.8|     0.075|                0.048|       

-------------------------------------------
Batch: 4
-------------------------------------------
+-----------+-------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|           residual|        prediction|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+-------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|16733.09091|  256.8568186495613|16476.234091350438|      13.46|    87.8|     0.072|                0.033|        0.152| 25185.27449| 13993.07536|    4|   1|
|9566.659857| 1560.0940893603201|  8006.56576763968|      19.61|    74.2|     0.338|                0.117|        0.148| 21262.30088| 10975.88358|    9|   6|
|28683.63636| 1082.5992657332063|27601.037094266794|      18.17|    73.0|     0.079|                11.67|       

...

-------------------------------------------
Batch: 15
-------------------------------------------
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|          residual|        prediction|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
| 11537.5076| 1713.658432846014| 9823.849167153987|      21.14|    72.7|     0.281|                0.051|        0.144| 25976.71772| 15255.18672|   10|   4|
|14235.01508| 885.2039052299533|13349.811174770046|      6.706|    89.8|     0.071|                 0.07|        0.119| 22332.20339| 13539.20973|    2|   3|
|29696.80251|3123.4476183637234|26573.354891636278|      28.44|   65.92|     4.904|                225.8|        186.

-------------------------------------------
Batch: 16
-------------------------------------------
+-----------+-------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|           residual|        prediction|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+-------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|10259.27711|  307.8551020267314|  9951.42200797327|       11.4|    80.3|     4.919|                297.2|        45.16| 25624.61538| 17959.09091|   11|  10|
|14228.46395| -4664.700648845055|18893.164598845055|      19.04|    79.2|      4.91|                3.575|        2.782| 25367.01443| 15673.49525|    8|   6|
|18507.63636|-1116.7863550818001|  19624.4227150818|      21.26|   43.45|     0.086|                784.0|      

-------------------------------------------
Batch: 17
-------------------------------------------
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|          residual|        prediction|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
| 16567.9598|-2872.786066241748| 19440.74586624175|      12.24|    78.8|     0.076|                63.47|        62.31| 34724.74576| 21151.36778|    2|  17|
|11525.83587|1810.9729950744331| 9714.862874925568|      22.01|   66.94|     4.924|                311.0|        43.35|  28018.5558| 21334.85477|   10|  16|
|26408.72727|1895.1212164061217|24513.606053593878|      16.18|    79.4|     0.072|                 83.6|         87.

-------------------------------------------
Batch: 18
-------------------------------------------
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|          residual|        prediction|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|15656.72727|275.67026403901036|15381.057005960989|      16.18|    86.3|     0.072|                0.044|        0.204| 23635.22067| 12651.32383|    4|   3|
|16828.58934|-4885.153462730224|21713.742802730223|      29.41|   52.25|     4.923|                340.9|        142.6| 34048.56826| 20394.93136|    8|  17|
|23115.48387|1083.4512172394861|22032.032652760514|      16.61|    83.3|     4.909|                0.026|        0.14

-------------------------------------------
Batch: 19
-------------------------------------------
+-----------+-------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|           residual|        prediction|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+-------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|14991.29724| 1494.4018582987756|13496.895381701224|      24.48|   55.35|     0.286|                464.9|        125.3| 33808.14159| 18984.19958|    9|  10|
|15765.66627| -202.0895660095648|15967.755836009565|       12.9|   69.69|     0.076|                 0.07|        0.141| 36422.81369|  30447.3765|   12|  21|
|25199.27638|  805.4247924022602| 24393.85158759774|      13.78|    72.1|     0.072|                0.051|      

## **Producing the Data**
At this point, the `power_streaming_data.csv` file should be uploaded into the main directory in `jupyterhub`. Additionally, the `Loring_FinalProject.py` file reads in the `power_streaming_data.csv` file and writes a loop that randomly samples 5 rows from this file, outputs them to a `.csv` (removing the indices), and pauses for 30 seconds in between outputting of datasets. This process occurs 20 times.

After running the above line of code to start the query, we can run the loop from the `Loring_FinalProject.py` file in a python console or terminal. Once we do this, we will see output here in this notebook right above the start of this section. Then the query will be stopped. Please see the video uploaded to Moodle within the Final Project assignment to see a recording of this process play out in real time.

**Note:** The project instructions state to use a pause time of 10 seconds, but class discussion has revealed that a longer pause helps prevent lagging and ensures all 20 files are created and output. Additionally, since the `label` join key may not be unique, some files may have more than 5 total rows, but there will be 5 distinct rows. Finally, the red line that appears above each batch in the notebook *not* representative of an actual error, but rather staging outputs from the log.

In [27]:
query.stop()